# German Credit Data: выбор модели для прогноза риска

Цель notebook: сравнить три подхода для прогноза `Risk`:

- линейная модель: `LogisticRegression`;
- дерево решений: `DecisionTreeClassifier`;
- метод ближайших соседей: `KNeighborsClassifier`.

В датасете целевая переменная `Risk` имеет значения `good` и `bad`, поэтому это задача классификации, а не обычной линейной регрессии. Для такой задачи корректная линейная модель — логистическая регрессия.

In [30]:
import warnings
warnings.filterwarnings("ignore")

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
)

## 1. Загрузка данных

In [31]:
data_path = r"C:\Users\Vitaliy\OneDrive\career\it_career_hub\Python Analytics\data\german_credit_data.csv"

df = pd.read_csv(data_path)
df.head()

,Unnamed: 0,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,Risk
0,0,67,male,2,own,NaN,little,1169,6,radio/TV,good
1,1,22,female,2,own,little,moderate,5951,48,radio/TV,bad
2,2,49,male,1,own,little,NaN,2096,12,education,good
3,3,45,male,2,free,little,little,7882,42,furniture/equipment,good
4,4,53,male,2,free,little,little,4870,24,car,bad


In [32]:
print("Размер данных:", df.shape)
display(df.info())
display(df.describe(include="all").T)

Размер данных: (1000, 11)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 11 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Unnamed: 0        1000 non-null   int64 
 1   Age               1000 non-null   int64 
 2   Sex               1000 non-null   object
 3   Job               1000 non-null   int64 
 4   Housing           1000 non-null   object
 5   Saving accounts   817 non-null    object
 6   Checking account  606 non-null    object
 7   Credit amount     1000 non-null   int64 
 8   Duration          1000 non-null   int64 
 9   Purpose           1000 non-null   object
 10  Risk              1000 non-null   object
dtypes: int64(5), object(6)
memory usage: 86.1+ KB


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,1000.0,NaN,NaN,NaN,499.5,288.819436,0.0,249.75,499.5,749.25,999.0
Age,1000.0,NaN,NaN,NaN,35.546,11.375469,19.0,27.0,33.0,42.0,75.0
Sex,1000,2,male,690,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Job,1000.0,NaN,NaN,NaN,1.904,0.653614,0.0,2.0,2.0,2.0,3.0
Housing,1000,3,own,713,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Saving accounts,817,4,little,603,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Checking account,606,3,little,274,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Credit amount,1000.0,NaN,NaN,NaN,3271.258,2822.736876,250.0,1365.5,2319.5,3972.25,18424.0
Duration,1000.0,NaN,NaN,NaN,20.903,12.058814,4.0,12.0,18.0,24.0,72.0
Purpose,1000,8,car,337,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 2. Проверка целевой переменной

In [33]:
df["Risk"].value_counts(normalize=True).rename("share").to_frame()

,share
Risk,
good,0.7
bad,0.3


Класс `bad` важен для бизнеса: ошибочно выдать кредит рискованному клиенту обычно хуже, чем отказать хорошему клиенту. Поэтому кроме `accuracy` будем смотреть:

- `recall_bad`: сколько плохих кредитов модель смогла найти;
- `precision_bad`: насколько точны прогнозы класса `bad`;
- `f1_bad`: баланс precision и recall для класса `bad`;
- `roc_auc`: качество ранжирования риска.

## 3. Подготовка признаков

In [34]:
df_model = df.copy()

# Удаляем технический индекс, который не должен помогать модели.
if "Unnamed: 0" in df_model.columns:
    df_model = df_model.drop(columns=["Unnamed: 0"])

X = df_model.drop(columns=["Risk"])
y = df_model["Risk"].map({"good": 0, "bad": 1})

numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

print("Числовые признаки:", numeric_features)
print("Категориальные признаки:", categorical_features)

Числовые признаки: ['Age', 'Job', 'Credit amount', 'Duration']
Категориальные признаки: ['Sex', 'Housing', 'Saving accounts', 'Checking account', 'Purpose']


In [35]:
numeric_scaled = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_preprocess = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocess_scaled = ColumnTransformer(
    transformers=[
        ("num", numeric_scaled, numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)

preprocess_tree = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_features),
        ("cat", categorical_preprocess, categorical_features),
    ]
)

## 4. Модели для сравнения

In [36]:
models = {
    "Logistic Regression": Pipeline(
        steps=[
            ("preprocess", preprocess_scaled),
            ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
        ]
    ),
    "Decision Tree": Pipeline(
        steps=[
            ("preprocess", preprocess_tree),
            ("model", DecisionTreeClassifier(
                max_depth=4,
                min_samples_leaf=20,
                class_weight="balanced",
                random_state=42,
            )),
        ]
    ),
    "KNN": Pipeline(
        steps=[
            ("preprocess", preprocess_scaled),
            ("model", KNeighborsClassifier(n_neighbors=15)),
        ]
    ),
}

## 5. Сравнение через cross-validation

In [37]:
scoring = {
    "accuracy": "accuracy",
    "precision_bad": "precision",
    "recall_bad": "recall",
    "f1_bad": "f1",
    "roc_auc": "roc_auc",
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for name, model in models.items():
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring)
    row = {"model": name}
    for metric in scoring:
        row[metric] = scores[f"test_{metric}"].mean()
    rows.append(row)

cv_results = (
    pd.DataFrame(rows)
    .set_index("model")
    .sort_values("f1_bad", ascending=False)
)

cv_results.round(3)

,accuracy,precision_bad,recall_bad,f1_bad,roc_auc
model,,,,,
Logistic Regression,0.636,0.426,0.617,0.504,0.689
Decision Tree,0.638,0.423,0.533,0.466,0.614
KNN,0.722,0.590,0.203,0.301,0.675


## 6. Проверка на hold-out test set

In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

test_rows = []
fitted_models = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    fitted_models[name] = model

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    test_rows.append(
        {
            "model": name,
            "accuracy": accuracy_score(y_test, y_pred),
            "precision_bad": precision_score(y_test, y_pred),
            "recall_bad": recall_score(y_test, y_pred),
            "f1_bad": f1_score(y_test, y_pred),
            "roc_auc": roc_auc_score(y_test, y_proba),
        }
    )

test_results = (
    pd.DataFrame(test_rows)
    .set_index("model")
    .sort_values("f1_bad", ascending=False)
)

test_results.round(3)

,accuracy,precision_bad,recall_bad,f1_bad,roc_auc
model,,,,,
Decision Tree,0.530,0.353,0.683,0.466,0.591
Logistic Regression,0.615,0.398,0.550,0.462,0.637
KNN,0.715,0.579,0.183,0.278,0.667


In [39]:
for name, model in fitted_models.items():
    y_pred = model.predict(X_test)
    print("=" * 80)
    print(name)
    print("Confusion matrix:")
    print(confusion_matrix(y_test, y_pred))
    print()
    print(classification_report(y_test, y_pred, target_names=["good", "bad"]))

Logistic Regression
Confusion matrix:
[[90 50]
 [27 33]]

              precision    recall  f1-score   support

        good       0.77      0.64      0.70       140
         bad       0.40      0.55      0.46        60

    accuracy                           0.61       200
   macro avg       0.58      0.60      0.58       200
weighted avg       0.66      0.61      0.63       200

Decision Tree
Confusion matrix:
[[65 75]
 [19 41]]

              precision    recall  f1-score   support

        good       0.77      0.46      0.58       140
         bad       0.35      0.68      0.47        60

    accuracy                           0.53       200
   macro avg       0.56      0.57      0.52       200
weighted avg       0.65      0.53      0.55       200

KNN
Confusion matrix:
[[132   8]
 [ 49  11]]

              precision    recall  f1-score   support

        good       0.73      0.94      0.82       140
         bad       0.58      0.18      0.28        60

    accuracy              

## 7. Итоговый выбор модели

In [40]:
best_model_name = cv_results["f1_bad"].idxmax()
best_model = fitted_models[best_model_name]

print(f"Лучшая модель по F1 для класса bad: {best_model_name}")
display(cv_results.round(3))

Лучшая модель по F1 для класса bad: Logistic Regression


,accuracy,precision_bad,recall_bad,f1_bad,roc_auc
model,,,,,
Logistic Regression,0.636,0.426,0.617,0.504,0.689
Decision Tree,0.638,0.423,0.533,0.466,0.614
KNN,0.722,0.590,0.203,0.301,0.675


## Вывод

Для этого датасета лучше всего подходит **Logistic Regression**.

Причины:

- это задача классификации `good/bad`, поэтому логистическая регрессия является правильной линейной моделью;
- по cross-validation она дает лучший `f1_bad`, то есть лучший баланс между поиском плохих кредитов и точностью таких прогнозов;
- KNN показывает неплохую `accuracy`, но хуже находит класс `bad`, поэтому для кредитного риска он менее полезен;
- дерево решений проще интерпретировать, но в данном сравнении уступает логистической регрессии по устойчивому качеству.

Финальная рекомендация: использовать **Logistic Regression** как базовую модель для прогноза риска, а дальше улучшать ее подбором гиперпараметров и настройкой threshold для класса `bad`.

## 8. Подбор гиперпараметров для лучшей модели

Гиперпараметры — это настройки модели, которые выбирает аналитик до обучения.

Для `LogisticRegression` один из главных гиперпараметров — `C`.

- маленький `C`: модель осторожнее, сильнее регуляризация;
- большой `C`: модель больше подстраивается под данные;
- слишком большой `C` может привести к переобучению.

Подберем `C` через `GridSearchCV` и снова оценим качество.

In [41]:
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
}

grid_search = GridSearchCV(
    estimator=models["Logistic Regression"],
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Лучшие параметры:", grid_search.best_params_)
print(f"Лучший CV F1 для класса bad: {grid_search.best_score_:.3f}")

tuned_model = grid_search.best_estimator_

Лучшие параметры: {'model__C': 0.1}
Лучший CV F1 для класса bad: 0.514


In [42]:
tuned_pred = tuned_model.predict(X_test)
tuned_proba = tuned_model.predict_proba(X_test)[:, 1]

tuned_results = pd.DataFrame(
    [
        {
            "model": "Tuned Logistic Regression",
            "accuracy": accuracy_score(y_test, tuned_pred),
            "precision_bad": precision_score(y_test, tuned_pred),
            "recall_bad": recall_score(y_test, tuned_pred),
            "f1_bad": f1_score(y_test, tuned_pred),
            "roc_auc": roc_auc_score(y_test, tuned_proba),
        }
    ]
).set_index("model")

tuned_results.round(3)

,accuracy,precision_bad,recall_bad,f1_bad,roc_auc
model,,,,,
Tuned Logistic Regression,0.62,0.405,0.567,0.472,0.65


После подбора гиперпараметров можно использовать `tuned_model` как финальную модель.

Если результат почти не изменился, это нормально: значит базовая логистическая регрессия уже была хорошей отправной точкой.

## 9. Как показывать прогноз

Прогноз модели состоит из двух частей:

- `predict()` возвращает итоговый класс: `good` или `bad`;
- `predict_proba()` возвращает вероятность класса `bad`.

В реальной задаче обычно важнее показывать не только класс, но и вероятность риска. Например:

- `0.12` — низкий риск;
- `0.48` — средний риск;
- `0.81` — высокий риск.

### 9.1. Прогноз для нескольких клиентов из test set

In [43]:
prediction_table = X_test.head(10).copy()

prediction_table["true_risk"] = y_test.head(10).map({0: "good", 1: "bad"}).values
prediction_table["probability_bad"] = tuned_model.predict_proba(X_test.head(10))[:, 1]
prediction_table["risk_bad_percent"] = (prediction_table["probability_bad"] * 100).round(1).astype(str) + "%"
prediction_table["predicted_risk"] = tuned_model.predict(X_test.head(10))
prediction_table["predicted_risk"] = prediction_table["predicted_risk"].map({0: "good", 1: "bad"})

prediction_table = prediction_table.sort_values("probability_bad", ascending=False)
prediction_table.drop(columns=["probability_bad"])

,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose,true_risk,risk_bad_percent,predicted_risk
112,28,male,1,rent,little,moderate,6260,18,car,good,65.7%,bad
537,37,female,2,own,little,moderate,3612,18,furniture/equipment,good,60.8%,bad
128,34,male,3,own,little,moderate,1860,12,car,good,43.9%,good
966,23,male,1,own,quite rich,moderate,2520,27,radio/TV,bad,42.7%,good
47,23,female,0,rent,quite rich,little,1352,6,car,good,41.7%,good
346,23,male,2,own,little,moderate,882,13,radio/TV,good,40.7%,good
216,31,male,2,own,little,little,3104,18,business,good,38.5%,good
289,48,male,2,own,little,little,1024,24,radio/TV,bad,35.0%,good
875,40,female,2,own,rich,moderate,1322,11,car,good,35.0%,good
30,36,male,2,own,rich,moderate,1913,18,business,good,32.8%,good


### 9.2. Прогноз для одной новой заявки

In [44]:
new_client = pd.DataFrame(
    [
        {
            "Age": 35,
            "Sex": "male",
            "Job": 2,
            "Housing": "own",
            "Saving accounts": "little",
            "Checking account": "moderate",
            "Credit amount": 3500,
            "Duration": 24,
            "Purpose": "car",
        }
    ]
)

probability_bad = tuned_model.predict_proba(new_client)[0, 1]
prediction = tuned_model.predict(new_client)[0]

print("Прогноз:", "bad" if prediction == 1 else "good")
print(f"Вероятность bad: {probability_bad:.3f}")
print(f"Риск bad в процентах: {probability_bad * 100:.1f}%")

new_client

Прогноз: bad
Вероятность bad: 0.569
Риск bad в процентах: 56.9%


,Age,Sex,Job,Housing,Saving accounts,Checking account,Credit amount,Duration,Purpose
0,35,male,2,own,little,moderate,3500,24,car


### Как это объяснять

Модель не “угадывает будущее” на 100%. Она оценивает риск по похожим клиентам из исторических данных.

Например, если `risk_bad_percent = 68.0%`, это значит: по признакам этой заявки модель считает риск плохого кредита высоким. Банк может решить:

- автоматически одобрять заявки с низкой вероятностью `bad`;
- отправлять средние случаи на ручную проверку;
- отклонять или дополнительно проверять заявки с высокой вероятностью `bad`.